In [1]:
import torch
from datasets import load_dataset

ds = load_dataset("nyu-mll/glue", "mnli")["validation_matched"]

/home/dwithun/Development/experiment/decomposition_xprmnt/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from neural_decomp import ModelManagementInterface, DeviceMapOptions
model_id = "google/gemma-3-1b-it"
mmi = ModelManagementInterface(model_id=model_id, precision=torch.float32, device_map=DeviceMapOptions.AUTO)
model = mmi.get_model()
tokenizer = mmi.get_tokenizer()
model

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 340/340 [00:00<00:00, 362.31it/s]


Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((1152,), e

In [3]:
model.model.layers

ModuleList(
  (0-25): 26 x Gemma3DecoderLayer(
    (self_attn): Gemma3Attention(
      (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
      (k_proj): Linear(in_features=1152, out_features=256, bias=False)
      (v_proj): Linear(in_features=1152, out_features=256, bias=False)
      (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
      (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
      (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
    )
    (mlp): Gemma3MLP(
      (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
      (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
      (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
      (act_fn): GELUTanh()
    )
    (input_layernorm): Gemma3RMSNorm((1152,), eps=1e-06)
    (post_attention_layernorm): Gemma3RMSNorm((1152,), eps=1e-06)
    (pre_feedforward_layernorm): Gemma3RMSNorm((1152,), eps=1e-06)
    (post_feedforward_layernorm): Gemma3RMSNorm((1152,), eps=

In [4]:
target_layer = model.model.layers[0]
benchmark_activations = {
    name: [] for name, _ in target_layer.named_modules() if name != ""
}

current_step_acts = {}

def capture_hook(submodule_name):
    def hook(module, input, output):
        act = output[0] if isinstance(output, tuple) else output
        current_step_acts[submodule_name] = act.detach().cpu()
    return hook

handles = []

for name, submodule in target_layer.named_modules():
    if name != "":
        h = submodule.register_forward_hook(capture_hook(name))
        handles.append(h)


In [5]:
label_names = ["entailment", "neutral", "contradiction"]
label_token_ids = [tokenizer.encode(" "+ name, add_special_tokens=False)[0] for name in label_names]
eval_data = ds.select(range(100))

In [6]:
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score

predictions = []
ground_truth = []

model.eval()
with torch.no_grad():
    for sample in tqdm(eval_data, desc="Evaluating on MNLI"):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship is entailment, neutral or contradiction.\n"
            f"Answer with one word:<end_of_turn>\n"
            f"<start_of_turn>model\n"
        )

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model(**inputs)

        for name, act_tensor in current_step_acts.items():
            sample_summary = act_tensor.squeeze(0).mean(dim=0)
            benchmark_activations[name].append(sample_summary)

        next_token_logits = outputs.logits[0, -1, :]
        candidate_logits = next_token_logits[label_token_ids]

        pred_label = torch.argmax(candidate_logits).item()
        predictions.append(pred_label)
        ground_truth.append(sample["label"])

for h in handles:
    h.remove()

accuracy = accuracy_score(ground_truth, predictions)
print(f"\nFinal Accuracy: {accuracy * 100:.2f}%")
print(
    "Classification report:\n"
    f"{classification_report(ground_truth, predictions, labels=[0, 1, 2], target_names=label_names, zero_division=0)}"
)


Evaluation on MLNI:   0%|          | 0/100 [00:00<?, ?it/s]


AttributeError: 'BaseModelOutputWithPast' object has no attribute 'detach'

In [ ]:
import numpy as np

for name in benchmark_activations:
    pooled_samples = []
    for item in benchmark_activations[name]:
        arr = item.cpu().numpy() if isinstance(item, torch.Tensor) else np.array(item)

        if arr.ndim == 2:
            pooled = arr.mean(axis=0)
        elif arr.ndim == 3:
            pooled = arr.mean(axis=1).reshape(-1)
        else:
            pooled = arr
        
        pooled_samples.append(pooled)
    benchmark_activations[name] = np.stack(pooled_samples)

for name, tensor in benchmark_activations.items():
    print(f"Submodule: {name:20} | Output Shape: {tensor.shape}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

# 1. Get MLP activations across benchmark samples
acts = benchmark_activations["mlp.act_fn"]
neuron_profiles = acts.T  # (6912, num_samples)

# 2. Top K most dynamic neurons
neuron_variance = np.var(neuron_profiles, axis=1)
top_k = 250
top_indices = np.argsort(neuron_variance)[-top_k:]
active_profiles = neuron_profiles[top_indices]

# 3. Normalize for pattern correlation
active_profiles_norm = normalize(active_profiles, norm='l2', axis=1)

# 4. Cluster into functional groups
num_clusters = 5
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init='auto')
cluster_labels = kmeans.fit_predict(active_profiles_norm)

# 5. Sort and visualize with Heatmap
sorted_order = np.argsort(cluster_labels)
sorted_matrix = active_profiles[sorted_order]
sorted_clusters = cluster_labels[sorted_order]

plt.figure(figsize=(12, 8))
sns.heatmap(sorted_matrix, cmap='magma', cbar_kws={'label': 'Activation Magnitude'})
plt.title(f"Top {top_k} Active Neurons Grouped into {num_clusters} Co-Activation Clusters")
plt.xlabel("Benchmark Sample Index (0 to 100)")
plt.ylabel("Active Neurons (Sorted by Cluster)")

cluster_boundaries = np.where(np.diff(sorted_clusters))[0]
for boundary in cluster_boundaries:
    plt.axhline(boundary + 1, color='cyan', linestyle='--', linewidth=1.5)

plt.show()
